# Step 4.1 Update Reveiw table with Bronze Column Matches

PURPOSE
* compare the candidate tokens extracted from the RAND HRS Codebook with the actual column names in your Bronze Delta table:
    * dev_catalog.brz_raw_hrs.randhrs1992_2022v1
* Update **bronze_column_match**---in table--->**hrs_variable_review**

This gives us an independent piece of evidence to help distinguish actual source variables from ordinary words, response codes, and other tokens. 

It will not change your existing classifications or approve/reject any candidates.

### Configuration

In [0]:
# ---------------------------------------------------------------
# 1. Configure the source tables 
# 
#    and confirm availability.
# ---------------------------------------------------------------
from pyspark.sql import functions as F

# Source Bronze table containing the ingested RAND HRS data
bronze_table = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

# Persistent Codebook candidate review table
review_table = "dev_catalog.hrs_agent_metadata.hrs_variable_review"

# Codebook version being evaluated
codebook_version = "1992-2022v1"

print(f"Bronze table: {bronze_table}")
print(f"Review table: {review_table}")
print(f"Codebook version: {codebook_version}")


### Read the Bronze colmn Names

In [0]:
# ---------------------------------------------------------------
# 2. Read the Bronze column names
# ---------------------------------------------------------------

# Retrieve the actual column names from the Bronze Delta table.
bronze_columns = spark.table(bronze_table).columns

# Build a Spark DataFrame containing each actual Bronze column.
bronze_columns_df = spark.createDataFrame(
    [(column_name,) for column_name in bronze_columns],
    ["bronze_column"]
).withColumn(
    "normalized_column",
    F.upper(F.trim(F.col("bronze_column")))
).dropDuplicates(["normalized_column"])

print(f"Number of Bronze columns: {len(bronze_columns)}")

display(
    bronze_columns_df
    .select("bronze_column")
    .orderBy("bronze_column")
)


### Compare candidate tokens against Bronze columns

MERGE writes the Boolean match result to your persistent review table. 
Because match_updates contains one row per normalized token and Codebook version, multiple candidate records containing the same token receive the same match result.

The merge intentionally does not change candidate_class, review_status, review_notes, or any business approval fields.

In [0]:

# ---------------------------------------------------------------
# 3. Compare candidate tokens against Bronze columns
#    and persist bronze_column_match in hrs_variable_review
# ---------------------------------------------------------------

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load candidates for the selected Codebook version.
review_df = (
    spark.table(review_table)
    .filter(F.col("codebook_version") == codebook_version)
    .filter(F.col("candidate_token").isNotNull())
    .filter(F.trim(F.col("candidate_token")) != "")
)

# Normalize Bronze column names and remove duplicates.
bronze_lookup = (
    bronze_columns_df
    .select(
        F.upper(F.trim(F.col("normalized_column"))).alias(
            "normalized_column"
        ),
        F.col("bronze_column")
    )
    .filter(F.col("normalized_column").isNotNull())
    .dropDuplicates(["normalized_column"])
)

# Summarize evidence for each unique candidate token.
candidate_summary = (
    review_df
    .withColumn(
        "normalized_token",
        F.upper(F.trim(F.col("candidate_token")))
    )
    .groupBy("normalized_token")
    .agg(
        F.first("candidate_token").alias("candidate_token"),
        F.count("*").alias("candidate_record_count"),
        F.countDistinct("source_page").alias("distinct_page_count"),
        F.min("source_page").alias("first_page"),
        F.max("source_page").alias("last_page"),
        F.sort_array(
            F.collect_set("candidate_class")
        ).alias("candidate_classes"),
        F.first(
            "context_excerpt", ignorenulls=True
        ).alias("example_context")
    )
)

# Exact, case-insensitive comparison with Bronze columns.
diagnostic_df = (
    candidate_summary
    .join(
        F.broadcast(bronze_lookup),
        candidate_summary.normalized_token
        == bronze_lookup.normalized_column,
        "left"
    )
    .withColumn(
        "bronze_column_match",
        F.col("bronze_column").isNotNull()
    )
    .withColumn(
        "bronze_match_status",
        F.when(
            F.col("bronze_column_match"),
            F.lit("Exact Bronze column match")
        ).otherwise(
            F.lit("No exact Bronze column match")
        )
    )
    .select(
        "normalized_token",
        "candidate_token",
        "bronze_column_match",
        "bronze_match_status",
        "bronze_column",
        "candidate_record_count",
        "distinct_page_count",
        "first_page",
        "last_page",
        "candidate_classes",
        "example_context"
    )
)



### Update the bronze_column_match field.

In [0]:
# 3.1 Persist one match result per normalized token.
# This updates bronze_column_match only.
# Classifications, review statuses, notes, and approvals are unchanged.

match_updates = (
    diagnostic_df
    .select(
        "normalized_token",
        "bronze_column_match"
    )
    .dropDuplicates(["normalized_token"])
    .withColumn(
        "codebook_version",
        F.lit(codebook_version)
    )
)

target = DeltaTable.forName(spark, review_table)

(
    target.alias("t")
    .merge(
        match_updates.alias("s"),
        """
        t.codebook_version = s.codebook_version
        AND UPPER(TRIM(t.candidate_token)) = s.normalized_token
        """
    )
    .whenMatchedUpdate(
        set={
            "bronze_column_match": "s.bronze_column_match"
        }
    )
    .execute()
)

# Create a temporary view for diagnostic queries.
diagnostic_df.createOrReplaceTempView(
    "hrs_candidate_bronze_diagnostic"
)

print("Candidate-to-Bronze comparison completed.")
print("bronze_column_match was updated for matching candidate records.")
print("Candidate classifications, review statuses, and approvals were not changed.")


In [0]:
# ---------------------------------------------------------------
# 4. Review the match counts
# ---------------------------------------------------------------

display(
    diagnostic_df
    .groupBy("bronze_match_status")
    .agg(
        F.count("*").alias("unique_candidate_tokens"),
        F.sum("candidate_record_count").alias(
            "candidate_records"
        )
    )
    .orderBy("bronze_match_status")
)


In [0]:

# ---------------------------------------------------------------
# 5. Inspect candidates matching Bronze columns
#    An exact match confirms source-column existence only.
# ---------------------------------------------------------------

display(
    diagnostic_df
    .filter(F.col("bronze_column_match") == True)
    .orderBy(
        F.desc("candidate_record_count"),
        "candidate_token"
    )
)


In [0]:

# ---------------------------------------------------------------
# 6. Inspect candidates without an exact Bronze column match
# ---------------------------------------------------------------

display(
    diagnostic_df
    .filter(F.col("bronze_column_match") == False)
    .orderBy(
        F.desc("candidate_record_count"),
        "candidate_token"
    )
)


In [0]:

# ---------------------------------------------------------------
# 7. Compare candidate classifications with Bronze match results
# ---------------------------------------------------------------

record_diagnostic = (
    spark.table(review_table)
    .filter(F.col("codebook_version") == codebook_version)
    .filter(F.col("candidate_token").isNotNull())
    .filter(F.trim(F.col("candidate_token")) != "")
)

display(
    record_diagnostic
    .groupBy("candidate_class", "bronze_column_match")
    .agg(
        F.count("*").alias("candidate_records"),
        F.countDistinct("candidate_token").alias(
            "distinct_tokens"
        )
    )
    .orderBy(
        "candidate_class",
        "bronze_column_match"
    )
)


In [0]:
# ---------------------------------------------------------------
# 8. Inspect the most important examples

# Helps explain how candidate classifications and Bronze matches behave for representative tokens.
#
# Select the review table columns and the matched Bronze column immediately after the join, 
# then use those selected names in the display query.
#
# First, use the diagnostic to establish these rules:
# - Exact match + Likely variable: strong candidate for subsequent Codebook validation.
# - Exact match + Needs review: investigate whether the naming heuristic is too restrictive.
# - No match + Needs review: inspect the Codebook context before deciding.
# - No match + Likely variable: check whether the variable is absent from the ingested source
# Once we see the cross-tabulation from Cell 7, we can revise the classifier using evidence instead of making assumptions. 
# The final approval process should still require Codebook validation and human review.
# ---------------------------------------------------------------

tokens_to_inspect = [
    "HRS", "NR", "DK", "RF", "W14", "AHEAD",
    "RAND", "AGE", "ADL", "IADL", "SSI",
    "SSDI", "IMPFLAG", "R10BMI", "R11BMI"
]

display(
    record_diagnostic
    .join(
        F.broadcast(bronze_lookup),
        F.upper(F.trim(F.col("candidate_token"))) == F.col("normalized_column"),
        "left"
    )
    .filter(
        F.upper(F.trim(F.col("candidate_token"))).isin(tokens_to_inspect)
    )
    .select(
        "candidate_token",
        "source_page",
        "candidate_class",
        "review_status",
        F.col("bronze_column").alias("matched_bronze_column"),
        "bronze_column_match",
        "context_excerpt"
    )
    .orderBy("candidate_token", "source_page")
)


In [0]:
# 9. Run a focused diagnostic for BMI candidates
# Checks the 16 BMI variables directly relevant to your use case.

# Verify BMI candidate evidence.
# Read-only diagnostic. Does not update persistent tables.

bmi_tokens = [
    "R1BMI", "R2BMI", "R3BMI", "R4BMI", "R5BMI",
    "R6BMI", "R7BMI", "R8BMI", "R9BMI", "R10BMI",
    "R11BMI", "R12BMI", "R13BMI", "R14BMI",
    "R15BMI", "R16BMI"
]

display(
    record_diagnostic
    .join(
        F.broadcast(bronze_lookup),
        F.upper(F.trim(F.col("candidate_token"))) == F.col("normalized_column"),
        "left"
    )
    .select(
        F.upper(F.trim(F.col("candidate_token"))).alias("candidate_token"),
        F.col("source_page"),
        F.col("candidate_class"),
        F.col("review_status"),
        F.col("bronze_column").alias("matched_bronze_column"),
        F.col("context_excerpt")
    )
    .filter(F.upper(F.trim(F.col("candidate_token"))).isin(bmi_tokens))
    .orderBy("candidate_token", "source_page")
)



In [0]:
# -----------------------------------------------------------
# 10 — Inspect likely variables without a Bronze match
#
# Finds possible Codebook variables missing from the ingested source or incorrectly classified by the naming heuristic.
# 
# What we're looking for: candidates that might be legitimate Codebook variables 
# but differ from the Bronze column name, as well as tokens that were incorrectly classified as variables.
#--------------------------------------------------------------

# Cell 10 — Inspect likely-variable candidates without exact Bronze matches.
# Read-only diagnostic. Does not update persistent tables.

display(
    record_diagnostic
    .join(
        F.broadcast(bronze_lookup),
        F.upper(F.trim(F.col("candidate_token"))) == F.col("normalized_column"),
        "left"
    )
    .withColumn(
        "bronze_match_status",
        F.when(
            F.col("bronze_column").isNotNull(),
            F.lit("Exact Bronze column match")
        ).otherwise(
            F.lit("No exact Bronze column match")
        )
    )
    .filter(
        (F.col("candidate_class") == "Likely variable") &
        (F.col("bronze_match_status") == "No exact Bronze column match")
    )
    .select(
        F.upper(F.trim(F.col("candidate_token"))).alias("candidate_token"),
        F.col("source_page"),
        F.col("candidate_class"),
        F.col("review_status"),
        F.col("bronze_column").alias("matched_bronze_column"),
        F.col("context_excerpt")
    )
    .orderBy("candidate_token", "source_page")
    .limit(100)
)
